# Modul 9: Morfologi

Nama: Sahrul Ramadhani  
NIM: 244107020058  
Kelas: TI 3C

## D. Tugas praktikum

### 1. Menghubungkan Google Drive

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    drive.mount('/content/drive')
    IMAGES = Path('/content/drive/MyDrive/PCVK/Minggu 9/images')
else:
    lokasi = [Path.cwd() / 'images', Path.cwd() / 'Minggu 9' / 'images']
    IMAGES = Path(os.environ['PCVK_IMAGES']) if os.environ.get('PCVK_IMAGES') else next(
        (folder for folder in lokasi if folder.is_dir()), lokasi[0]
    )

HASIL = Path.cwd() / 'hasil_minggu9'
HASIL.mkdir(parents=True, exist_ok=True)
print('Folder gambar:', IMAGES)

### 2. Import library

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from skimage import filters, io, color, morphology, img_as_float
from skimage.morphology import skeletonize

plt.rcParams['figure.dpi'] = 100

def baca_gambar(nama, grayscale=False):
    path = IMAGES / nama
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE if grayscale else cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f'Gambar tidak ditemukan atau tidak dapat dibaca: {path}')
    return image

def tampilkan(images, titles, rows=1, cols=None, figsize=(12, 4), cmaps=None, simpan=None):
    cols = len(images) if cols is None else cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    for i, (ax, image, title) in enumerate(zip(axes.flat, images, titles)):
        if image.ndim == 2:
            ax.imshow(image, cmap=cmaps[i] if cmaps else 'gray')
        else:
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    for ax in list(axes.flat)[len(images):]:
        ax.axis('off')
    fig.tight_layout()
    if simpan:
        fig.savefig(HASIL / simpan, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

gambar_wajib = ['plat nomer.jpg', 'j.png', 'kitten01.jpg', 'lily.jpg']
belum_ada = [nama for nama in gambar_wajib if not (IMAGES / nama).is_file()]
if belum_ada:
    raise FileNotFoundError('Gambar belum tersedia: ' + ', '.join(belum_ada))

plat_gray = baca_gambar('plat nomer.jpg', grayscale=True)
plat_bgr = baca_gambar('plat nomer.jpg')

### 3. Dilasi manual dengan cross 3 x 3

In [ ]:
def dilasi_citra(F, SED):
    p, q = F.shape
    tinggi, lebar = SED.shape
    cy, cx = tinggi // 2, lebar // 2
    padded = np.pad(F, ((cy, cy), (cx, cx)), mode='constant', constant_values=0)
    imgD = np.zeros((p, q), dtype=np.uint8)
    aktif = SED != 0
    for i in range(p):
        for j in range(q):
            temp = padded[i:i + tinggi, j:j + lebar]
            imgD[i, j] = np.max(temp[aktif])
    return imgD

SED_cross3 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=np.uint8)
ret, thresh_d3 = cv2.threshold(plat_gray, 125, 200, cv2.THRESH_BINARY)
imgD_manual = dilasi_citra(thresh_d3, SED_cross3)
print(SED_cross3)
tampilkan([plat_gray, imgD_manual], ['Citra awal', 'Dilasi manual, cross 3 x 3'])

### 4. Dilasi OpenCV dengan cross 3 x 3

In [ ]:
ret, thresh_d4 = cv2.threshold(plat_bgr, 127, 225, cv2.THRESH_BINARY)
kernel_cross3 = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
dilasi_cv = cv2.dilate(thresh_d4, kernel_cross3, iterations=1)
print(kernel_cross3)
tampilkan([plat_bgr, dilasi_cv], ['Citra awal', 'Dilasi OpenCV, cross 3 x 3'])

### 5. Erosi manual dengan square 5 x 5

In [ ]:
def erosi_citra(F, SE):
    m, n = F.shape
    tinggi, lebar = SE.shape
    cy, cx = tinggi // 2, lebar // 2
    padded = np.pad(F, ((cy, cy), (cx, cx)), mode='constant', constant_values=255)
    imgE = np.zeros((m, n), dtype=np.uint8)
    aktif = SE != 0
    for i in range(m):
        for j in range(n):
            temp = padded[i:i + tinggi, j:j + lebar]
            imgE[i, j] = np.min(temp[aktif])
    return imgE

SE_square5 = np.ones((5, 5), dtype=np.uint8)
ret1, thresh_d5 = cv2.threshold(plat_gray, 175, 225, cv2.THRESH_BINARY)
imgE_manual = erosi_citra(thresh_d5, SE_square5)
tampilkan([plat_gray, imgE_manual], ['Citra awal', 'Erosi manual, square 5 x 5'])

### 6. Erosi OpenCV dengan square 5 x 5

In [ ]:
kernel_square5 = np.ones((5, 5), np.uint8)
erosion_cv = cv2.erode(plat_bgr, kernel_square5, iterations=1)
tampilkan([plat_bgr, erosion_cv], ['Citra awal', 'Erosi OpenCV, square 5 x 5'])

### 7. Opening dengan square 3 x 3

In [ ]:
kernel_square3 = np.ones((3, 3), np.uint8)
ret, thresh_morfologi = cv2.threshold(plat_gray, 127, 225, cv2.THRESH_BINARY)
opening_manual = dilasi_citra(erosi_citra(thresh_morfologi, kernel_square3), kernel_square3)
tampilkan([plat_gray, opening_manual], ['Citra awal', 'Opening manual, square 3 x 3'])

In [ ]:
opening_cv = cv2.morphologyEx(thresh_morfologi, cv2.MORPH_OPEN, kernel_square3)
tampilkan(
    [plat_gray, thresh_morfologi, opening_cv],
    ['Citra awal', 'Citra biner', 'Opening OpenCV, square 3 x 3']
)

### 8. Closing dengan square 3 x 3

In [ ]:
closing_manual = erosi_citra(dilasi_citra(thresh_morfologi, kernel_square3), kernel_square3)
tampilkan([plat_gray, closing_manual], ['Citra awal', 'Closing manual, square 3 x 3'])

In [ ]:
closing_cv = cv2.morphologyEx(thresh_morfologi, cv2.MORPH_CLOSE, kernel_square3)
tampilkan(
    [plat_gray, thresh_morfologi, closing_cv],
    ['Citra awal', 'Citra biner', 'Closing OpenCV, square 3 x 3']
)

### 9. Dilasi dengan cross 3 x 3 dan 5 x 5

In [ ]:
SED_cross5 = np.array([
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 1, 1, 1, 1],
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
], dtype=np.uint8)
dilasi_cross = {}
for ukuran, kernel in [(3, SED_cross3), (5, SED_cross5)]:
    dilasi_cross[ukuran] = dilasi_citra(thresh_morfologi, kernel)
    print(f'SED cross {ukuran} x {ukuran}:')
    print(kernel)
    tampilkan([plat_gray, dilasi_cross[ukuran]], ['Citra awal', f'Dilasi cross {ukuran} x {ukuran}'])

### 10. Dilasi dengan circular 3 x 3 dan 5 x 5

In [ ]:
dilasi_circular = {}
kernel_circular = {}
for ukuran in [3, 5]:
    kernel_circular[ukuran] = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ukuran, ukuran))
    dilasi_circular[ukuran] = dilasi_citra(thresh_morfologi, kernel_circular[ukuran])
    print(f'SED circular {ukuran} x {ukuran}:')
    print(kernel_circular[ukuran])
    tampilkan([plat_gray, dilasi_circular[ukuran]], ['Citra awal', f'Dilasi circular {ukuran} x {ukuran}'])

### 11. Erosi dengan rectangle 3 x 5 dan 5 x 7

In [ ]:
erosi_rectangle = {}
kernel_rectangle = {}
for tinggi, lebar in [(3, 5), (5, 7)]:
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (lebar, tinggi))
    kernel_rectangle[(tinggi, lebar)] = kernel
    erosi_rectangle[(tinggi, lebar)] = erosi_citra(thresh_morfologi, kernel)
    print(f'SE rectangle {tinggi} x {lebar}:')
    print(kernel)
    tampilkan(
        [plat_gray, erosi_rectangle[(tinggi, lebar)]],
        ['Citra awal', f'Erosi rectangle {tinggi} x {lebar}']
    )

### 12. Erosi dengan line vertikal 3 dan 5 pada j.png

In [ ]:
img_j = baca_gambar('j.png', grayscale=True)
ret, thresh_j = cv2.threshold(img_j, 127, 225, cv2.THRESH_BINARY)
erosi_vertikal = {}
kernel_vertikal = {}
for panjang in [3, 5]:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (1, panjang))
    kernel_vertikal[panjang] = kernel
    erosi_vertikal[panjang] = erosi_citra(thresh_j, kernel)
    print(f'SE line vertikal {panjang}:')
    print(kernel)
    tampilkan([img_j, erosi_vertikal[panjang]], ['Citra awal', f'Erosi line vertikal {panjang}'])

### 13. TopHat

In [ ]:
filterSize = (3, 3)
kernel_tophat = cv2.getStructuringElement(cv2.MORPH_RECT, filterSize)
img = plat_gray.copy()
tophat_img = cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel_tophat)
kitten_gray = cv2.cvtColor(baca_gambar('kitten01.jpg'), cv2.COLOR_BGR2GRAY)
tophat_kitten = cv2.morphologyEx(kitten_gray, cv2.MORPH_TOPHAT, kernel_tophat)
tampilkan(
    [img, tophat_img, kitten_gray, tophat_kitten],
    ['Plat nomor', 'TopHat plat, 3 x 3', 'Kitten', 'TopHat kitten, 3 x 3'],
    rows=2, cols=2, figsize=(12, 8)
)

### 14. BlackHat

In [ ]:
Kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
blackhat = cv2.morphologyEx(kitten_gray, cv2.MORPH_BLACKHAT, Kernel)
tampilkan([kitten_gray, blackhat], ['Citra awal', 'BlackHat, kernel lebar 13 dan tinggi 5'])

### 15. Skeleton

In [ ]:
img_lily = baca_gambar('lily.jpg', grayscale=True)
binary_lily = img_lily > filters.threshold_triangle(img_lily)
skeleton_lily = skeletonize(binary_lily)
tampilkan(
    [img_lily, skeleton_lily], ['Original image', 'Skeleton image'],
    figsize=(12, 6), simpan='binary and bin_skeleton.png'
)

### 16. Skeleton inverse

In [ ]:
img_lily_red = io.imread(IMAGES / 'lily.jpg')[..., 0]
binary_lily_inverse = img_lily_red < filters.threshold_triangle(img_lily_red)
skeleton_lily_inverse = skeletonize(binary_lily_inverse)
tampilkan(
    [img_lily_red, skeleton_lily_inverse], ['Original', 'Skeleton (inverse)'],
    figsize=(12, 6), cmaps=['gray', 'gray_r']
)

### 17. Thickening

Kode ini memakai `thin` untuk menipiskan objek.

In [ ]:
image = img_as_float(color.rgb2gray(io.imread(IMAGES / 'lily.jpg')))
image_binary = image < 0.5
out_skeletonize = morphology.skeletonize(image_binary)
out_thin = morphology.thin(image_binary)
tampilkan([image, out_thin], ['Input', 'Thin'], figsize=(10, 4), simpan='char_out.png')

### Analisis hasil percobaan

- Dilasi memperbesar objek putih. Erosi memperkecilnya.
- Opening adalah erosi lalu dilasi. Closing adalah dilasi lalu erosi.
- Pada OpenCV, cross dan circular 3 x 3 memiliki susunan piksel yang sama. Pada ukuran 5 x 5, susunannya berbeda.
- Rectangle lebih memengaruhi arah sisi panjangnya. Line vertikal hanya memeriksa tetangga di atas dan bawah.
- TopHat mengambil detail terang yang lebih kecil dari kernel. BlackHat mengambil detail gelap yang lebih kecil dari kernel.
- Skeleton dan thin menipiskan objek. Membalik syarat threshold mengubah bagian gambar yang dianggap sebagai objek.

## E. Tugas praktikum: morfologi pada dataset KTP

Tulisan gelap diubah menjadi objek putih sebelum proses morfologi. Saat ditampilkan, warnanya dibalik lagi menjadi tulisan hitam pada latar putih.

In [ ]:
file_ktp = sorted(
    (p for p in IMAGES.iterdir() if p.is_file() and p.stem.lower().startswith('ktp')
     and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}),
    key=lambda p: p.name.lower()
)
if not file_ktp:
    raise FileNotFoundError(f'Gambar KTP belum tersedia di {IMAGES}')

dataset_ktp = {}
for path in file_ktp:
    citra = baca_gambar(path.name)
    if citra.shape[0] > citra.shape[1]:
        citra = cv2.rotate(citra, cv2.ROTATE_90_COUNTERCLOCKWISE)
    dataset_ktp[path.name] = citra
print('Jumlah citra KTP:', len(dataset_ktp))

### E.1. Perbandingan operasi morfologi

In [ ]:
hasil_ktp = {}
for nama, citra in dataset_ktp.items():
    gray = cv2.cvtColor(citra, cv2.COLOR_BGR2GRAY)
    nilai_otsu, biner = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((3, 3), np.uint8)
    erosi = cv2.erode(biner, kernel, iterations=1)
    dilasi = cv2.dilate(biner, kernel, iterations=1)
    opening = cv2.morphologyEx(biner, cv2.MORPH_OPEN, kernel)
    closing = cv2.morphologyEx(biner, cv2.MORPH_CLOSE, kernel)
    kernel_hat = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
    top = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel_hat)
    black = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel_hat)
    skeleton = skeletonize(biner > 0)
    thin = morphology.thin(biner > 0)
    gradient = cv2.morphologyEx(gray, cv2.MORPH_GRADIENT, kernel)
    smoothing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel)
    hasil_ktp[nama] = {
        'gray': gray, 'threshold': nilai_otsu, 'biner': biner,
        'erosi': erosi, 'dilasi': dilasi, 'opening': opening, 'closing': closing,
        'tophat': top, 'blackhat': black, 'skeleton': skeleton, 'thin': thin,
        'gradient': gradient, 'smoothing': smoothing,
    }
    print(nama, '| threshold Otsu:', nilai_otsu)
    tampilkan(
        [gray, 255 - biner, 255 - erosi, 255 - dilasi, 255 - opening, 255 - closing,
         top, black, ~skeleton, ~thin, gradient, 255 - smoothing],
        ['Grayscale', 'Biner', 'Erosi 3 x 3', 'Dilasi 3 x 3', 'Opening 3 x 3', 'Closing 3 x 3',
         'TopHat 13 x 5', 'BlackHat 13 x 5', 'Skeleton', 'Thin', 'Gradient 3 x 3', 'Opening lalu closing'],
        rows=4, cols=3, figsize=(16, 16)
    )

### E.2. Perbandingan bentuk dan ukuran kernel

In [ ]:
kernel_uji = {
    'Rectangle 1 x 2': cv2.getStructuringElement(cv2.MORPH_RECT, (2, 1)),
    'Line vertikal 3': cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3)),
    'Square 2 x 2': cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2)),
    'Cross 3 x 3': cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3)),
    'Circular 5 x 5': cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)),
    'Rectangle 3 x 5': cv2.getStructuringElement(cv2.MORPH_RECT, (5, 3)),
}
for nama, hasil in hasil_ktp.items():
    hasil['uji_kernel'] = {
        label: cv2.morphologyEx(hasil['biner'], cv2.MORPH_CLOSE, kernel)
        for label, kernel in kernel_uji.items()
    }
    print(nama)
    tampilkan(
        [255 - image for image in hasil['uji_kernel'].values()],
        ['Closing: ' + label for label in hasil['uji_kernel']],
        rows=3, cols=2, figsize=(14, 12)
    )

### E.3. Perbaikan latar dan keterbacaan teks

In [ ]:
for nama, hasil in hasil_ktp.items():
    gray = hasil['gray']
    gray_halus = cv2.GaussianBlur(gray, (3, 3), 0)
    perbaikan = {}
    for lebar, tinggi in [(13, 5), (21, 9), (31, 13)]:
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (lebar, tinggi))
        latar = cv2.morphologyEx(gray_halus, cv2.MORPH_CLOSE, kernel)
        normal = cv2.divide(gray_halus, np.maximum(latar, 1), scale=255)
        nilai, teks = cv2.threshold(normal, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        perbaikan[(lebar, tinggi)] = {'latar': latar, 'normal': normal, 'teks': teks, 'threshold': nilai}
    hasil['perbaikan'] = perbaikan
    print(nama)
    tampilkan(
        [255 - hasil['biner']] + [255 - p['teks'] for p in perbaikan.values()],
        ['Biner awal', 'Perbaikan latar 13 x 5', 'Perbaikan latar 21 x 9', 'Perbaikan latar 31 x 13'],
        rows=2, cols=2, figsize=(14, 9)
    )
    teks = perbaikan[(21, 9)]['teks']
    kernel_teks = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 1))
    teks_rapi = cv2.morphologyEx(teks, cv2.MORPH_CLOSE, kernel_teks)
    hasil['hasil_akhir'] = 255 - teks_rapi
    tampilkan(
        [dataset_ktp[nama], hasil['hasil_akhir']],
        ['KTP asli', 'Perbaikan latar 21 x 9 dan closing 1 x 2'],
        rows=2, cols=1, figsize=(14, 13)
    )

### Analisis tugas KTP

- Dilasi menyambung tulisan yang putus. Kernel terlalu besar bisa membuat huruf yang berdekatan menyatu.
- Erosi menipiskan objek putih. Erosi dan opening bisa menghapus bagian huruf yang tipis.
- Closing menutup celah kecil. Ukuran kernel disesuaikan dengan ketebalan dan jarak huruf.
- TopHat mengambil detail terang. BlackHat mengambil detail gelap, seperti tulisan. Hasilnya bergantung pada ukuran kernel.
- Closing pada gambar abu-abu memperkirakan latar. Membagi gambar dengan latar ini meratakan terang sebelum gambar diubah menjadi hitam putih.
- Gaussian 3 x 3 mengurangi pola kecil yang mengganggu tulisan sebelum perbaikan latar.
- Skeleton dan thin mengambil bentuk dasar huruf dengan garis tipis. Hasilnya belum tentu lebih mudah dibaca.
- Hasil akhir memakai perbaikan latar 21 x 9 dan closing 1 x 2. Ketajaman foto, bayangan, dan ukuran huruf tetap memengaruhi hasilnya.

## Kesimpulan

Morfologi mengubah bentuk objek berdasarkan kernel. Dilasi menambah bagian objek, erosi menguranginya, opening menghapus bagian kecil, dan closing menutup celah. Pada KTP, ukuran kernel harus cukup kecil agar bagian huruf tetap terjaga.